# RLHF: Reward Model + PPO Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Synthetic Preference Data

In production, human annotators create preference data. We'll create synthetic pairs where the "preferred" response is objectively better (more concise, more accurate, more helpful).

In [ ]:
```python

import numpy as np

PREFERENCE_DATA = [

    {

        "prompt": "What is the capital of France?",

        "preferred": "The capital of France is Paris.",

        "rejected": "France is a country in Europe. It has many cities. The capital is Paris. Paris is known for the Eiffel Tower.",

    },

    {

        "prompt": "Explain gravity in one sentence.",

        "preferred": "Gravity is the force that attracts objects with mass toward each other.",

        "rejected": "Gravity is something that makes things fall down when you drop them.",

    },

    {

        "prompt": "What is 15 times 7?",

        "preferred": "15 times 7 is 105.",

        "rejected": "Let me think about this. 15 times 7. Well, 10 times 7 is 70, and 5 times 7 is 35, so the answer might be around 105.",

    },

    {

        "prompt": "Name three programming languages.",

        "preferred": "Python, Rust, and TypeScript.",

        "rejected": "There are many programming languages. Some popular ones include various languages like Python and others.",

    },

    {

        "prompt": "What year did World War II end?",

        "preferred": "World War II ended in 1945.",

        "rejected": "World War II was a major global conflict. It involved many countries. The war ended in the mid-1940s, specifically in 1945.",

    },

    {

        "prompt": "Define machine learning.",

        "preferred": "Machine learning is a field where algorithms learn patterns from data to make predictions without being explicitly programmed.",

        "rejected": "Machine learning is a type of AI. AI stands for artificial intelligence. Machine learning uses data to learn.",

    },

]

In [ ]:
```

The preferred responses are concise and direct. The rejected responses exhibit common failure modes: unnecessary padding, hedging, redundant explanation, and imprecision. This is exactly the kind of distinction that SFT cannot capture but RLHF can.

### Step 2: Reward Model Architecture

The reward model reuses the transformer architecture from the mini GPT, but replaces the vocabulary-sized output head with a single scalar projection.

In [ ]:
```python

import sys

import os

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "..", "04-pre-training-mini-gpt", "code"))

from main import MiniGPT, LayerNorm, Embedding, TransformerBlock

class RewardModel:

    def __init__(self, vocab_size=256, embed_dim=128, num_heads=4,

                 num_layers=4, max_seq_len=128, ff_dim=512):

        self.embedding = Embedding(vocab_size, embed_dim, max_seq_len)

        self.blocks = [

            TransformerBlock(embed_dim, num_heads, ff_dim)

            for _ in range(num_layers)

        ]

        self.ln_f = LayerNorm(embed_dim)

        self.reward_head = np.random.randn(embed_dim) * 0.02

    def forward(self, token_ids):

        seq_len = token_ids.shape[-1]

        mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)

        x = self.embedding.forward(token_ids)

        for block in self.blocks:

            x = block.forward(x, mask)

        x = self.ln_f.forward(x)

        last_hidden = x[:, -1, :]

        reward = last_hidden @ self.reward_head

        return reward

In [ ]:
```

The reward model takes the hidden state at the *last* token position and projects it to a scalar. Why the last token? Because the causal attention mask means the last position has attended to every previous token. It has the most complete representation of the entire (prompt, response) sequence.

### Step 3: Bradley-Terry Loss

Train the reward model on preference pairs using the Bradley-Terry pairwise loss.

In [ ]:
```python

def tokenize_for_reward(prompt, response, vocab_size=256):

    prompt_tokens = [min(t, vocab_size - 1) for t in list(prompt.encode("utf-8"))]

    response_tokens = [min(t, vocab_size - 1) for t in list(response.encode("utf-8"))]

    return prompt_tokens + [0] + response_tokens

def sigmoid(x):

    return np.where(

        x >= 0,

        1.0 / (1.0 + np.exp(-x)),

        np.exp(x) / (1.0 + np.exp(x))

    )

def bradley_terry_loss(reward_preferred, reward_rejected):

    diff = reward_preferred - reward_rejected

    loss = -np.log(sigmoid(diff) + 1e-8)

    return loss

def train_reward_model(rm, preference_data, num_epochs=10, lr=1e-4, max_seq_len=128):

    print(f"Training Reward Model: {len(preference_data)} preference pairs, {num_epochs} epochs")

    print()

    losses = []

    accuracies = []

    for epoch in range(num_epochs):

        epoch_loss = 0.0

        epoch_correct = 0

        num_pairs = 0

        indices = np.random.permutation(len(preference_data))

        for idx in indices:

            pair = preference_data[idx]

            preferred_tokens = tokenize_for_reward(pair["prompt"], pair["preferred"])

            rejected_tokens = tokenize_for_reward(pair["prompt"], pair["rejected"])

            preferred_tokens = preferred_tokens[:max_seq_len]

            rejected_tokens = rejected_tokens[:max_seq_len]

            preferred_ids = np.array(preferred_tokens).reshape(1, -1)

            rejected_ids = np.array(rejected_tokens).reshape(1, -1)

            r_preferred = rm.forward(preferred_ids)[0]

            r_rejected = rm.forward(rejected_ids)[0]

            loss = bradley_terry_loss(r_preferred, r_rejected)

            if r_preferred > r_rejected:

                epoch_correct += 1

            diff = r_preferred - r_rejected

            grad = sigmoid(diff) - 1.0

            rm.reward_head -= lr * grad * rm.ln_f.forward(

                rm.embedding.forward(preferred_ids)

            )[:, -1, :].flatten()

            epoch_loss += loss

            num_pairs += 1

        avg_loss = epoch_loss / max(num_pairs, 1)

        accuracy = epoch_correct / max(num_pairs, 1)

        losses.append(avg_loss)

        accuracies.append(accuracy)

        if epoch % 2 == 0:

            print(f"  Epoch {epoch + 1:3d} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.1%}")

    return rm, losses, accuracies

In [ ]:
```

The accuracy metric is straightforward: what fraction of preference pairs does the reward model rank correctly? A random model scores 50%. A well-trained reward model on clean data should exceed 70%. InstructGPT's reward model achieved about 72% accuracy on held-out comparisons, which sounds low but is actually good -- many preference pairs are ambiguous even to humans (inter-annotator agreement was about 73%).

### Step 4: Simplified PPO Loop

Full PPO is complex. This implementation captures the core mechanism: generate responses, score them, compute the advantage, and update the policy with a KL penalty.

In [ ]:
```python

def compute_kl_divergence(policy_logits, reference_logits):

    policy_probs = np.exp(policy_logits - policy_logits.max(axis=-1, keepdims=True))

    policy_probs = policy_probs / policy_probs.sum(axis=-1, keepdims=True)

    policy_probs = np.clip(policy_probs, 1e-10, 1.0)

    ref_probs = np.exp(reference_logits - reference_logits.max(axis=-1, keepdims=True))

    ref_probs = ref_probs / ref_probs.sum(axis=-1, keepdims=True)

    ref_probs = np.clip(ref_probs, 1e-10, 1.0)

    kl = np.sum(policy_probs * np.log(policy_probs / ref_probs), axis=-1)

    return kl.mean()

def generate_response(model, prompt_tokens, max_new_tokens=30, temperature=0.8, max_seq_len=128):

    tokens = list(prompt_tokens)

    for _ in range(max_new_tokens):

        context = np.array(tokens[-max_seq_len:]).reshape(1, -1)

        logits = model.forward(context)

        next_logits = logits[0, -1, :]

        next_logits = next_logits / max(temperature, 1e-8)

        probs = np.exp(next_logits - next_logits.max())

        probs = probs / probs.sum()

        probs = np.clip(probs, 1e-10, 1.0)

        probs = probs / probs.sum()

        next_token = np.random.choice(len(probs), p=probs)

        tokens.append(int(next_token))

    return tokens

def copy_model_weights(source, target):

    target.embedding.token_embed = source.embedding.token_embed.copy()

    target.embedding.pos_embed = source.embedding.pos_embed.copy()

    target.ln_f.gamma = source.ln_f.gamma.copy()

    target.ln_f.beta = source.ln_f.beta.copy()

    for s_block, t_block in zip(source.blocks, target.blocks):

        t_block.attn.W_q = s_block.attn.W_q.copy()

        t_block.attn.W_k = s_block.attn.W_k.copy()

        t_block.attn.W_v = s_block.attn.W_v.copy()

        t_block.attn.W_out = s_block.attn.W_out.copy()

        t_block.ffn.W1 = s_block.ffn.W1.copy()

        t_block.ffn.W2 = s_block.ffn.W2.copy()

        t_block.ffn.b1 = s_block.ffn.b1.copy()

        t_block.ffn.b2 = s_block.ffn.b2.copy()

        t_block.ln1.gamma = s_block.ln1.gamma.copy()

        t_block.ln1.beta = s_block.ln1.beta.copy()

        t_block.ln2.gamma = s_block.ln2.gamma.copy()

        t_block.ln2.beta = s_block.ln2.beta.copy()

def ppo_training(policy_model, reference_model, reward_model, prompts,

                 num_episodes=20, lr=1.5e-5, kl_coeff=0.02, max_seq_len=128):

    print(f"PPO Training: {num_episodes} episodes, lr={lr}, KL coeff={kl_coeff}")

    print()

    rewards_history = []

    kl_history = []

    for episode in range(num_episodes):

        prompt_text = prompts[episode % len(prompts)]

        prompt_tokens = [min(t, 252) for t in list(prompt_text.encode("utf-8"))]

        response_tokens = generate_response(

            policy_model, prompt_tokens,

            max_new_tokens=20, temperature=0.8, max_seq_len=max_seq_len

        )

        response_ids = np.array(response_tokens[:max_seq_len]).reshape(1, -1)

        reward = reward_model.forward(response_ids)[0]

        policy_logits = policy_model.forward(response_ids)

        ref_logits = reference_model.forward(response_ids)

        kl = compute_kl_divergence(policy_logits, ref_logits)

        total_reward = reward - kl_coeff * kl

        rewards_history.append(float(reward))

        kl_history.append(float(kl))

        for block in policy_model.blocks:

            update_scale = lr * total_reward

            block.ffn.W1 += update_scale * np.random.randn(*block.ffn.W1.shape) * 0.01

            block.ffn.W2 += update_scale * np.random.randn(*block.ffn.W2.shape) * 0.01

        if episode % 5 == 0:

            avg_reward = np.mean(rewards_history[-5:]) if rewards_history else 0

            avg_kl = np.mean(kl_history[-5:]) if kl_history else 0

            print(f"  Episode {episode:3d} | Reward: {reward:.4f} | KL: {kl:.4f} | "

                  f"Avg Reward: {avg_reward:.4f}")

    return policy_model, rewards_history, kl_history

In [ ]:
```

The core loop: (1) sample a prompt, (2) generate a response, (3) score it with the reward model, (4) compute KL divergence against the frozen reference, (5) compute the adjusted reward (reward minus KL penalty), (6) update the policy. The KL penalty grows as the policy diverges from the reference, automatically preventing reward hacking.

### Step 5: Reward Score Comparison

After RLHF, the policy model's responses should score higher on the reward model than the original SFT model's responses.

In [ ]:
```python

def compare_models(sft_model, rlhf_model, reward_model, prompts, max_seq_len=128):

    print("Model Comparison (reward scores)")

    print("-" * 60)

    print(f"  {'Prompt':<35} {'SFT':>10} {'RLHF':>10}")

    print("  " + "-" * 55)

    sft_total = 0.0

    rlhf_total = 0.0

    for prompt in prompts:

        prompt_tokens = [min(t, 252) for t in list(prompt.encode("utf-8"))]

        sft_response = generate_response(

            sft_model, prompt_tokens,

            max_new_tokens=20, temperature=0.6, max_seq_len=max_seq_len

        )

        rlhf_response = generate_response(

            rlhf_model, prompt_tokens,

            max_new_tokens=20, temperature=0.6, max_seq_len=max_seq_len

        )

        sft_ids = np.array(sft_response[:max_seq_len]).reshape(1, -1)

        rlhf_ids = np.array(rlhf_response[:max_seq_len]).reshape(1, -1)

        sft_reward = reward_model.forward(sft_ids)[0]

        rlhf_reward = reward_model.forward(rlhf_ids)[0]

        sft_total += sft_reward

        rlhf_total += rlhf_reward

        truncated_prompt = prompt[:33] + ".." if len(prompt) > 35 else prompt

        print(f"  {truncated_prompt:<35} {sft_reward:>10.4f} {rlhf_reward:>10.4f}")

    n = len(prompts)

    print("  " + "-" * 55)

    print(f"  {'Average':<35} {sft_total/n:>10.4f} {rlhf_total/n:>10.4f}")

    return sft_total / n, rlhf_total / n

In [ ]:
```

## Exercises

In [ ]:
1. Modify the reward model to use the mean of all hidden states instead of just the last position. Compare accuracy. The mean pooling approach gives every token equal weight, while the last-position approach relies on the causal attention to aggregate information. Test on the 6 preference pairs and report which approach scores higher accuracy.

2. Implement reward model calibration. After training, run all preference pairs through the reward model and compute: (a) the average reward for preferred responses, (b) the average reward for rejected responses, (c) the margin (preferred minus rejected). A well-calibrated model should have a clear margin. Then add 4 new preference pairs and check if the margin holds on unseen data.

3. Simulate reward hacking. Create a reward model that gives high scores to long responses (reward = len(response) / 100). Run PPO with this flawed reward model and observe the policy model generating increasingly long, repetitive outputs. Then add a KL penalty of 0.1 and show that it prevents the degenerate behavior.

4. Implement a multi-objective reward. Train two reward models -- one for helpfulness and one for conciseness. Combine them as R = 0.7 * R_helpful + 0.3 * R_concise. Show that the combined objective produces responses that are both helpful and concise, avoiding the verbosity trap of a single helpfulness reward.

5. Compare different KL coefficients. Run PPO with beta=0.001 (too low, reward hacking), beta=0.02 (standard), and beta=0.5 (too high, no learning). Plot the reward curve and KL curve for each. The beta=0.02 run should show steady reward improvement with bounded KL.